# 16 — Action-conditioned video planning

**Lesson type: simulator analogue.** This notebook links rendered observations, action-conditioned prediction, and feedback planning. It uses a fixed image-moment encoder and a small polynomial model. It does **not** implement JEPA or reproduce V-JEPA 2-AC or PiJEPA.

## Learning outcome

- Extract a compact feature from two simulator frames.
- Fit a predictor that uses both the current feature and action.
- Check prediction under changed actions before checking closed-loop control.
- Compare uniform-start CEM, controller-prior-start CEM, and the direct controller.

The task is MountainCarContinuous. Pillow draws frames from the simulator's public observation. The predictor and planners receive only features extracted from those frames. The simulator state is used to create images and score real outcomes.

## Fixed evaluation protocol

- **Training**: random actions on reset seeds 1000–1019.
- **Interventions**: 20 held-out visual states from seeds 2000–2004. Test actions are -1, 0, and 1.
- **Closed-loop evaluation**: seeds 3000–3019, reused for all three methods.
- **Methods**: neutral-mean CEM; momentum-prior-mean CEM; direct visual momentum control.
- **Planner budget**: horizon 10, 24 candidates, 4 elites, and 2 CEM updates per decision. Both planners use the same limits and feedback rule.
- **Episode limit**: 300 steps. Each planner has a 60-second total evaluation cap. Stopped and incomplete episodes remain in the CSV.
- **Success**: Gymnasium reports termination at the goal.

Prediction error, changed-action predictions, task success, and planning cost answer different questions. A high one-step score does not prove that a planner is useful.

In [ ]:
import csv
import os
import platform
import time
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import PIL
import sklearn
from PIL import Image, ImageDraw
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures

GOAL_POSITION = 0.45
MAX_STEPS = 300
TRAIN_SEEDS = tuple(range(1000, 1020))
INTERVENTION_SEEDS = tuple(range(2000, 2005))
EVALUATION_SEEDS = tuple(range(3000, 3020))
ACTIONS = (-1.0, 0.0, 1.0)
CEM_HORIZON = 10
CEM_CANDIDATES = 24
CEM_ELITES = 4
CEM_ITERATIONS = 2
METHOD_CAP_SECONDS = 60.0

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Gymnasium:", gym.__version__)
print("Pillow:", PIL.__version__)
print("Matplotlib:", __import__("matplotlib").__version__)
print("Platform:", platform.platform())
print("CPU:", platform.processor() or platform.machine())
print("Device: CPU")
print("Train seeds:", TRAIN_SEEDS[0], "through", TRAIN_SEEDS[-1])
print("Intervention seeds:", INTERVENTION_SEEDS[0], "through", INTERVENTION_SEEDS[-1])
print("Evaluation seeds:", EVALUATION_SEEDS[0], "through", EVALUATION_SEEDS[-1])
print("CEM: horizon", CEM_HORIZON, "candidates", CEM_CANDIDATES,
      "elites", CEM_ELITES, "updates", CEM_ITERATIONS)

env = gym.make("MountainCarContinuous-v0", max_episode_steps=MAX_STEPS)
env.unwrapped.goal_position = GOAL_POSITION

## Turn frames into visual features

The renderer uses a red car and a green target. The encoder finds the horizontal center of each color. It estimates velocity from the car's movement between adjacent frames. This fixed visual feature is small and easy to inspect; it is not a learned video representation.

In [ ]:
IMAGE_WIDTH = 256
IMAGE_HEIGHT = 96
X_MIN_PIXEL = 8
X_MAX_PIXEL = IMAGE_WIDTH - 8
MIN_POSITION = -1.2
MAX_POSITION = 0.6
MAX_SPEED = 0.07
X_PIXELS, Y_PIXELS = np.meshgrid(
    np.arange(IMAGE_WIDTH, dtype=np.float64),
    np.arange(IMAGE_HEIGHT, dtype=np.float64),
)


def position_to_pixel(position):
    fraction = (float(position) - MIN_POSITION) / (MAX_POSITION - MIN_POSITION)
    return X_MIN_PIXEL + np.clip(fraction, 0.0, 1.0) * (X_MAX_PIXEL - X_MIN_PIXEL)


def pixel_to_position(pixel):
    fraction = (float(pixel) - X_MIN_PIXEL) / (X_MAX_PIXEL - X_MIN_PIXEL)
    return MIN_POSITION + np.clip(fraction, 0.0, 1.0) * (MAX_POSITION - MIN_POSITION)


def track_height(position):
    return int(70 - 20 * np.sin(3.0 * float(position)))


background = Image.new("RGB", (IMAGE_WIDTH, IMAGE_HEIGHT), (18, 24, 34))
background_draw = ImageDraw.Draw(background)
track_points = [
    (x, track_height(pixel_to_position(x)))
    for x in range(X_MIN_PIXEL, X_MAX_PIXEL + 1, 2)
]
background_draw.line(track_points, fill=(150, 160, 170), width=2)
STATIC_BACKGROUND = np.asarray(background, dtype=np.uint8)


def render_frame(position):
    frame = STATIC_BACKGROUND.copy()
    car_x = position_to_pixel(position)
    car_y = track_height(position) - 7
    goal_x = position_to_pixel(GOAL_POSITION)
    goal_y = track_height(GOAL_POSITION) - 18
    car_blob = np.exp(-0.5 * (
        ((X_PIXELS - car_x) / 1.7) ** 2 + ((Y_PIXELS - car_y) / 1.7) ** 2
    ))
    goal_blob = np.exp(-0.5 * (
        ((X_PIXELS - goal_x) / 1.7) ** 2 + ((Y_PIXELS - goal_y) / 1.7) ** 2
    ))
    frame[..., 0] = np.clip(frame[..., 0].astype(np.float64) + 230 * car_blob,
                            0, 255).astype(np.uint8)
    frame[..., 1] = np.clip(frame[..., 1].astype(np.float64) + 230 * goal_blob,
                            0, 255).astype(np.uint8)
    return frame


def color_center_x(frame, color):
    red, green, blue = (frame[..., index].astype(np.float64) for index in range(3))
    if color == "red":
        weights = np.maximum(red - np.maximum(green, blue), 0.0)
    elif color == "green":
        weights = np.maximum(green - np.maximum(red, blue), 0.0)
    else:
        raise ValueError("color must be red or green")
    column_weights = weights.sum(axis=0)
    weight_sum = float(column_weights.sum())
    if weight_sum <= 0.0:
        raise ValueError(f"The rendered frame has no {color} marker")
    return float(np.dot(np.arange(frame.shape[1]), column_weights) / weight_sum)


def visual_feature(previous_frame, current_frame):
    previous_position = pixel_to_position(color_center_x(previous_frame, "red"))
    position = pixel_to_position(color_center_x(current_frame, "red"))
    goal_position = pixel_to_position(color_center_x(current_frame, "green"))
    velocity = np.clip(position - previous_position, -MAX_SPEED, MAX_SPEED)
    feature = np.asarray([position, velocity, goal_position], dtype=np.float64)
    if not np.isfinite(feature).all():
        raise ValueError("Visual feature contains a non-finite value")
    return feature

In [ ]:
example_previous = render_frame(-0.54)
example_current = render_frame(-0.50)
example_feature = visual_feature(example_previous, example_current)
print("Encoded [position, velocity, goal]:", example_feature)
assert abs(example_feature[0] - (-0.50)) < 0.01
assert abs(example_feature[1] - 0.04) < 0.015
assert abs(example_feature[2] - GOAL_POSITION) < 0.01

fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].imshow(example_previous)
axes[0].set_title("Previous frame")
axes[1].imshow(example_current)
axes[1].set_title("Current frame; red car, green goal")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Collect random-action transitions and fit a predictor

Training, intervention, and evaluation use different reset seeds. Each model input contains the feature recovered from images and the action that followed it. The target contains the next position and velocity recovered from the next pair of images.

In [ ]:
train_inputs = []
train_targets = []
training_episode_sizes = []

for seed in TRAIN_SEEDS:
    observation, _ = env.reset(seed=seed)
    previous_frame = render_frame(observation[0])
    current_frame = previous_frame
    rng = np.random.default_rng(seed + 50000)
    episode_size = 0

    for _ in range(MAX_STEPS):
        current_feature = visual_feature(previous_frame, current_frame)
        action = float(rng.uniform(-1.0, 1.0))
        next_observation, _, terminated, truncated, _ = env.step(
            np.asarray([action], dtype=np.float32)
        )
        next_frame = render_frame(next_observation[0])
        next_feature = visual_feature(current_frame, next_frame)
        train_inputs.append([*current_feature, action])
        train_targets.append(next_feature[:2].copy())
        previous_frame, current_frame = current_frame, next_frame
        episode_size += 1
        if terminated or truncated:
            break
    training_episode_sizes.append(episode_size)

train_inputs = np.asarray(train_inputs, dtype=np.float64)
train_targets = np.asarray(train_targets, dtype=np.float64)
poly = PolynomialFeatures(degree=3, include_bias=True)
train_design = poly.fit_transform(train_inputs)
regressor = Ridge(alpha=1e-6, fit_intercept=False)
regressor.fit(train_design, train_targets)
model_powers = poly.powers_.copy()
model_weights = regressor.coef_.T.copy()


def polynomial_features(inputs):
    inputs = np.asarray(inputs, dtype=np.float64)
    return np.prod(
        np.power(inputs[:, None, :], model_powers[None, :, :]), axis=2
    )


def learned_step(features, actions):
    features = np.asarray(features, dtype=np.float64)
    actions = np.asarray(actions, dtype=np.float64).reshape(-1)
    model_input = np.column_stack((features, actions))
    prediction = polynomial_features(model_input) @ model_weights
    position = np.clip(prediction[:, 0], MIN_POSITION, MAX_POSITION)
    velocity = np.clip(prediction[:, 1], -MAX_SPEED, MAX_SPEED)
    velocity = np.where((position == MIN_POSITION) & (velocity < 0), 0.0, velocity)
    return np.column_stack((position, velocity, features[:, 2]))

train_predictions = learned_step(train_inputs[:, :3], train_inputs[:, 3])
train_mae_position = float(np.mean(np.abs(train_predictions[:, 0] - train_targets[:, 0])))
train_mae_velocity = float(np.mean(np.abs(train_predictions[:, 1] - train_targets[:, 1])))
print("Training transitions:", len(train_inputs))
print("Training episodes:", training_episode_sizes)
print("Training MAE position:", train_mae_position)
print("Training MAE velocity:", train_mae_velocity)

## Measure held-out action interventions

For each held-out visual state, the simulator and model receive the same candidate actions. The model's next-feature error is separate from its ranking of those actions by a one-step goal utility. This checks whether the model responds to changed actions; it does not prove closed-loop success.

In [ ]:
heldout_states = []
for seed in INTERVENTION_SEEDS:
    observation, _ = env.reset(seed=seed)
    previous_frame = render_frame(observation[0])
    current_frame = previous_frame
    rng = np.random.default_rng(seed + 70000)
    path_features = []

    for _ in range(MAX_STEPS):
        path_features.append(visual_feature(previous_frame, current_frame))
        action = float(rng.uniform(-1.0, 1.0))
        next_observation, _, terminated, truncated, _ = env.step(
            np.asarray([action], dtype=np.float32)
        )
        next_frame = render_frame(next_observation[0])
        previous_frame, current_frame = current_frame, next_frame
        if terminated or truncated:
            break

    selected = np.linspace(0, len(path_features) - 1, 4, dtype=int)
    heldout_states.extend(path_features[index].copy() for index in selected)

heldout_states = np.asarray(heldout_states[:20], dtype=np.float64)
assert heldout_states.shape == (20, 3)

predicted_interventions = []
observed_interventions = []
intervention_records = []

def one_step_utility(features):
    features = np.asarray(features)
    return -((features[..., 0] - GOAL_POSITION) ** 2 + 0.05 * features[..., 1] ** 2)

for state_index, feature in enumerate(heldout_states):
    predictions = learned_step(
        np.repeat(feature[None, :], len(ACTIONS), axis=0), np.asarray(ACTIONS)
    )
    observations = []
    for action in ACTIONS:
        env.reset(seed=0)
        env.unwrapped.state = np.asarray(feature[:2], dtype=np.float32).copy()
        current_frame = render_frame(feature[0])
        next_observation, _, _, _, _ = env.step(
            np.asarray([action], dtype=np.float32)
        )
        next_frame = render_frame(next_observation[0])
        observed_feature = visual_feature(current_frame, next_frame)
        observations.append(observed_feature)
    observations = np.asarray(observations)
    predicted_interventions.append(predictions)
    observed_interventions.append(observations)

    for action_index, action in enumerate(ACTIONS):
        error = predictions[action_index, :2] - observations[action_index, :2]
        intervention_records.append({
            "state_index": state_index,
            "action": action,
            "predicted_position": float(predictions[action_index, 0]),
            "observed_position": float(observations[action_index, 0]),
            "predicted_velocity": float(predictions[action_index, 1]),
            "observed_velocity": float(observations[action_index, 1]),
            "absolute_position_error": float(abs(error[0])),
            "absolute_velocity_error": float(abs(error[1])),
            "predicted_utility": float(one_step_utility(predictions[action_index:action_index + 1])[0]),
            "observed_utility": float(one_step_utility(observations[action_index:action_index + 1])[0]),
        })

predicted_interventions = np.asarray(predicted_interventions)
observed_interventions = np.asarray(observed_interventions)
intervention_position_mae = float(np.mean(np.abs(
    predicted_interventions[..., 0] - observed_interventions[..., 0]
)))
intervention_velocity_mae = float(np.mean(np.abs(
    predicted_interventions[..., 1] - observed_interventions[..., 1]
)))
predicted_utility = one_step_utility(predicted_interventions)
observed_utility = one_step_utility(observed_interventions)
pairwise_correct = pairwise_total = excluded_ties = 0
for predicted_row, observed_row in zip(predicted_utility, observed_utility):
    for left in range(len(ACTIONS)):
        for right in range(left + 1, len(ACTIONS)):
            predicted_delta = predicted_row[left] - predicted_row[right]
            observed_delta = observed_row[left] - observed_row[right]
            if abs(predicted_delta) < 1e-8 or abs(observed_delta) < 1e-8:
                excluded_ties += 1
                continue
            pairwise_total += 1
            pairwise_correct += int(np.sign(predicted_delta) == np.sign(observed_delta))

print("Intervention states:", len(heldout_states))
print("Action cases:", len(intervention_records))
print("Held-out intervention MAE position:", intervention_position_mae)
print("Held-out intervention MAE velocity:", intervention_velocity_mae)
print("Non-tied pairwise utility ranking:", pairwise_correct, "/", pairwise_total)
print("Excluded tied action pairs:", excluded_ties)

intervention_path = Path("docs/evidence/action-video-planning-interventions.csv")
intervention_path.parent.mkdir(parents=True, exist_ok=True)
intervention_fields = list(intervention_records[0])
with intervention_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=intervention_fields)
    writer.writeheader()
    writer.writerows(intervention_records)
print("Wrote intervention evidence:", intervention_path)

## Plan from visual features

CEM samples bounded action sequences, scores their predicted feature rollouts, and executes only the first action. It replans after the next rendered observation. The policy prior is the simple momentum controller: it chooses the direction of current velocity. The prior-seeded planner starts its CEM mean from a sequence generated by that controller. It is not a learned VLA policy.

In [ ]:
def momentum_policy(feature):
    return 1.0 if float(feature[1]) >= 0.0 else -1.0


def policy_prior_sequence(start_feature):
    feature = np.asarray(start_feature, dtype=np.float64).copy()
    sequence = np.zeros(CEM_HORIZON, dtype=np.float64)
    calls = 0
    for step in range(CEM_HORIZON):
        sequence[step] = momentum_policy(feature)
        feature = learned_step(feature[None, :], np.asarray([sequence[step]]))[0]
        calls += 1
    return sequence, calls


def plan_with_cem(start_feature, rng, use_policy_prior):
    if use_policy_prior:
        mean, model_calls = policy_prior_sequence(start_feature)
    else:
        mean = np.zeros(CEM_HORIZON, dtype=np.float64)
        model_calls = 0
    standard_deviation = np.full(CEM_HORIZON, 0.8, dtype=np.float64)

    for _ in range(CEM_ITERATIONS):
        candidates = np.clip(
            rng.normal(mean, standard_deviation,
                       size=(CEM_CANDIDATES, CEM_HORIZON)),
            -1.0, 1.0,
        )
        features = np.repeat(np.asarray(start_feature)[None, :],
                             CEM_CANDIDATES, axis=0)
        costs = np.zeros(CEM_CANDIDATES, dtype=np.float64)
        for step in range(CEM_HORIZON):
            actions = candidates[:, step]
            features = learned_step(features, actions)
            model_calls += CEM_CANDIDATES
            costs += (
                (GOAL_POSITION - features[:, 0]) ** 2
                + 0.05 * features[:, 1] ** 2
                + 0.001 * actions ** 2
            )
        elite_indices = np.argsort(costs)[:CEM_ELITES]
        elites = candidates[elite_indices]
        mean = elites.mean(axis=0)
        standard_deviation = np.maximum(elites.std(axis=0), 0.05)

    return float(np.clip(mean[0], -1.0, 1.0)), model_calls

## Compare closed-loop outcomes

All methods use the same reset seeds and task limit. Each model planner uses the same fixed CEM budget. A 60-second method cap bounds total runtime; if it stops early, the remaining seed rows are kept with a stopped status. The direct controller's lower model-call count is reported as part of the comparison.

In [ ]:
METHODS = (
    "uniform_cem",
    "momentum_prior_cem",
    "direct_momentum",
)


def run_episode(method, seed, capture_visualization=False):
    observation, _ = env.reset(seed=seed)
    previous_frame = render_frame(observation[0])
    current_frame = previous_frame
    rng = np.random.default_rng(seed + 90000)
    episode_return = 0.0
    model_calls = 0
    planning_seconds = 0.0
    success = False
    steps = 0
    terminal_status = "step_limit"
    initial_feature = visual_feature(current_frame, current_frame)
    visual_history = [(0, current_frame.copy(), float(initial_feature[0]))] if capture_visualization else []

    for step in range(MAX_STEPS):
        feature = visual_feature(previous_frame, current_frame)
        if capture_visualization and step > 0 and step % 25 == 0:
            visual_history.append((step, current_frame.copy(), float(feature[0])))
        if method == "uniform_cem":
            started = time.perf_counter()
            action, calls = plan_with_cem(feature, rng, use_policy_prior=False)
            planning_seconds += time.perf_counter() - started
            model_calls += calls
        elif method == "momentum_prior_cem":
            started = time.perf_counter()
            action, calls = plan_with_cem(feature, rng, use_policy_prior=True)
            planning_seconds += time.perf_counter() - started
            model_calls += calls
        elif method == "direct_momentum":
            action = momentum_policy(feature)
        else:
            raise ValueError(f"unknown method: {method}")

        next_observation, reward, terminated, truncated, _ = env.step(
            np.asarray([action], dtype=np.float32)
        )
        next_frame = render_frame(next_observation[0])
        if capture_visualization and (terminated or truncated or step == MAX_STEPS - 1):
            final_feature = visual_feature(current_frame, next_frame)
            visual_history.append((step + 1, next_frame.copy(), float(final_feature[0])))
        previous_frame, current_frame = current_frame, next_frame
        episode_return += float(reward)
        steps = step + 1
        if terminated:
            success = True
            terminal_status = "goal"
            break
        if truncated:
            terminal_status = "simulator_time_limit"
            break

    record = {
        "method": method,
        "seed": int(seed),
        "status": "complete",
        "success": bool(success),
        "return": float(episode_return),
        "steps": int(steps),
        "planning_seconds": float(planning_seconds),
        "model_calls": int(model_calls),
        "terminal_status": terminal_status,
    }
    if capture_visualization:
        record["visual_history"] = visual_history
    return record


evaluation_records = []
method_runtime = {}
method_complete = {}
for method in METHODS:
    method_started = time.perf_counter()
    method_complete[method] = True
    for seed_index, seed in enumerate(EVALUATION_SEEDS):
        if time.perf_counter() - method_started >= METHOD_CAP_SECONDS:
            method_complete[method] = False
            for remaining_seed in EVALUATION_SEEDS[seed_index:]:
                evaluation_records.append({
                    "method": method,
                    "seed": int(remaining_seed),
                    "status": "stopped_time_cap",
                    "success": "",
                    "return": "",
                    "steps": 0,
                    "planning_seconds": 0.0,
                    "model_calls": 0,
                    "terminal_status": "not_started",
                })
            break
        evaluation_records.append(run_episode(method, seed))
    method_runtime[method] = time.perf_counter() - method_started

for method in METHODS:
    rows = [r for r in evaluation_records
            if r["method"] == method and r["status"] == "complete"]
    successes = sum(bool(r["success"]) for r in rows)
    print(method, "completed", len(rows), "successes", successes,
          "elapsed_s", round(method_runtime[method], 3),
          "within_cap", method_complete[method])

In [ ]:
def wilson_interval(successes, total, z=1.959963984540054):
    if total == 0:
        return float("nan"), float("nan")
    rate = successes / total
    denominator = 1.0 + z * z / total
    center = (rate + z * z / (2.0 * total)) / denominator
    margin = z * np.sqrt(
        rate * (1.0 - rate) / total + z * z / (4.0 * total * total)
    ) / denominator
    return max(0.0, center - margin), min(1.0, center + margin)


summary_rows = []
for method in METHODS:
    rows = [r for r in evaluation_records
            if r["method"] == method and r["status"] == "complete"]
    successes = sum(bool(r["success"]) for r in rows)
    low, high = wilson_interval(successes, len(rows))
    summary_rows.append({
        "method": method,
        "completed": len(rows),
        "successes": successes,
        "success_rate": successes / len(rows) if rows else float("nan"),
        "wilson_low": low,
        "wilson_high": high,
        "mean_return": float(np.mean([r["return"] for r in rows]))
            if rows else float("nan"),
        "mean_steps": float(np.mean([r["steps"] for r in rows]))
            if rows else float("nan"),
        "model_calls": int(sum(r["model_calls"] for r in rows)),
        "planning_seconds": float(sum(r["planning_seconds"] for r in rows)),
        "elapsed_seconds": float(method_runtime.get(method, 0.0)),
        "within_cap": bool(method_complete[method]),
    })

print("method | episodes | successes | success rate [95% Wilson] | mean return | mean steps | model calls | planning seconds")
for row in summary_rows:
    print(f"{row['method']} | {row['completed']} | {row['successes']} | "
          f"{row['success_rate']:.3f} [{row['wilson_low']:.3f}, {row['wilson_high']:.3f}] | "
          f"{row['mean_return']:.3f} | {row['mean_steps']:.1f} | "
          f"{row['model_calls']} | {row['planning_seconds']:.3f}")

records_path = Path("docs/evidence/action-video-planning-episodes.csv")
records_path.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ["method", "seed", "status", "success", "return", "steps",
              "planning_seconds", "model_calls", "terminal_status"]
with records_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(evaluation_records)
print("Wrote per-episode evidence:", records_path)

by_method_seed = {
    method: {
        row["seed"]: row for row in evaluation_records
        if row["method"] == method and row["status"] == "complete"
    }
    for method in METHODS
}
for planner_method in ("uniform_cem", "momentum_prior_cem"):
    paired = [
        (by_method_seed[planner_method][seed],
         by_method_seed["direct_momentum"][seed])
        for seed in EVALUATION_SEEDS
        if seed in by_method_seed[planner_method]
        and seed in by_method_seed["direct_momentum"]
    ]
    success_delta = np.asarray([
        int(left["success"]) - int(right["success"])
        for left, right in paired
    ], dtype=np.float64)
    return_delta = np.asarray([
        left["return"] - right["return"] for left, right in paired
    ], dtype=np.float64)
    step_delta = np.asarray([
        left["steps"] - right["steps"] for left, right in paired
    ], dtype=np.float64)
    print(
        f"Paired {planner_method} minus direct_momentum (n={len(paired)} "
        "evaluation starts; not training-seed uncertainty): "
        f"success={success_delta.mean():.3f}, "
        f"return={return_delta.mean():.3f}, steps={step_delta.mean():.1f}"
    )


In [ ]:
methods = [row["method"] for row in summary_rows]
rates = [row["success_rate"] for row in summary_rows]
errors = [
    [row["success_rate"] - row["wilson_low"],
     row["wilson_high"] - row["success_rate"]]
    for row in summary_rows
]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(methods, rates, color=["#64748b", "#2563eb", "#16a34a"])
if np.isfinite(np.asarray(errors)).all():
    axes[0].errorbar(methods, rates, yerr=np.asarray(errors).T,
                     fmt="none", color="black", capsize=4)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Goal success rate")
axes[0].tick_params(axis="x", rotation=20)
axes[0].set_title("Paired held-out starts")
axes[1].bar(methods, [row["planning_seconds"] for row in summary_rows],
            color=["#64748b", "#2563eb", "#16a34a"])
axes[1].set_ylabel("Total planning time (s)")
axes[1].tick_params(axis="x", rotation=20)
axes[1].set_title("Model-planner work")
plt.tight_layout()
plt.show()

# This single replay is for teaching only. It is not added to the 20-start metrics.
example_episode = run_episode("momentum_prior_cem", EVALUATION_SEEDS[0], capture_visualization=True)
visual_history = example_episode["visual_history"]
fig = plt.figure(figsize=(12, 5.5))
grid = fig.add_gridspec(2, 1, height_ratios=[1.25, 1.0], hspace=0.38)
image_grid = grid[0].subgridspec(1, len(visual_history), wspace=0.03)
for index, (step, frame, _) in enumerate(visual_history):
    image_ax = fig.add_subplot(image_grid[0, index])
    image_ax.imshow(frame)
    image_ax.set_title(f"Step {step}", fontsize=8)
    image_ax.axis("off")
path_ax = fig.add_subplot(grid[1])
path_ax.plot([item[0] for item in visual_history],
             [item[2] for item in visual_history], marker="o")
path_ax.axhline(GOAL_POSITION, color="green", linestyle="--", label="goal")
path_ax.set(xlabel="Episode step", ylabel="Position estimated from frames",
            title=f"Controller-prior CEM replay, seed {EVALUATION_SEEDS[0]}")
path_ax.legend()
fig.suptitle("Illustrative replay only; not included in the paired summary")
plt.show()

env.close()

## What the result can show

Use the outcome table and the per-episode CSV to inspect both successes and failures. The fixed feature extractor can recover a car position from these rendered frames. That does not show that a pretrained visual representation will preserve task-relevant state under camera changes or new objects.

The local prior is a hand-written momentum controller. It asks whether an informed action sequence can help a small CEM search in this task. It does not test the learned Octo policy prior or the MPPI implementation from PiJEPA. Results from this task cannot validate robot navigation or any paper's reported success rate.

## Research context

- Assran et al., [V-JEPA 2: Self-Supervised Video Models Enable Understanding, Prediction and Planning](https://arxiv.org/abs/2506.09985). The V-JEPA 2-AC system post-trains action-conditioned latent prediction and plans toward image goals. The [official implementation](https://github.com/facebookresearch/vjepa2) provides code and examples; this notebook uses neither its model nor its robot data.
- Chahe and Zhou, [Policy-Guided World Model Planning for Language-Conditioned Visual Navigation](https://arxiv.org/abs/2603.25981), CVPR 2026 workshop. PiJEPA uses a learned policy prior to warm-start MPPI over a JEPA world model. Its [authors' implementation](https://github.com/AmirhoseinCh/PiJEPA) requires CAST data and a much larger training and evaluation setup.
- Zhang et al., [V-JEPA Policy](https://arxiv.org/abs/2609.37250) (v2 dated 2026-10-10), and Lin et al., [JEPA-WAM](https://arxiv.org/abs/2608.09381) are newer world-action model directions. They couple predictive visual latents with action generation; this notebook does not.

## Check your understanding

1. Why does the predictor need both a visual feature and an action?
2. What can held-out action interventions show that one-step random-action error may miss?
3. Why must a planner's imagined success be checked in the simulator?
4. Which changes would be needed before this lesson could count as a paper reproduction?